# Challenge 5 (bonus): Deploy the answer workflow to Agent Platform

Deploys the Challenge 4 agent (Greeter → Search → Critique → Refine graph workflow) to **Agent Platform** (Agent Engine), then tests the deployed copy.

1. **Define** the agent: the Challenge 4 code, unchanged.
2. **Test locally** by wrapping it in `AdkApp`, the same wrapper the platform runs.
3. **Deploy** with `client.agent_engines.create(...)`.
4. **Test remotely** by streaming a query to the deployed agent.

**To run:** Runtime → Run all. Deployment takes several minutes. The project is detected from the environment; a staging bucket `gs://<project>-agent-staging` is created if it doesn't exist.

In [ ]:
%pip install -q "google-cloud-aiplatform[agent_engines,adk]" "google-adk==2.7.0"
# If this upgraded packages that were already imported, restart the runtime before continuing.

## Configuration

In [ ]:
import logging
import os
import subprocess

MODEL = "gemini-2.5-flash"
GCP_LOCATION = "global"                 # Gemini on Agent Platform: global endpoint, more capacity than one region


def detect_project_id() -> str:
    """Finds the active Google Cloud project from the environment or gcloud."""
    project = os.environ.get("GOOGLE_CLOUD_PROJECT")
    if not project:
        try:
            import google.auth

            _, project = google.auth.default()
        except Exception:
            project = None
    if not project:
        result = subprocess.run(
            ["gcloud", "config", "get-value", "project"], capture_output=True, text=True
        )
        project = result.stdout.strip()
    if not project:
        raise RuntimeError("No Google Cloud project found. Set GOOGLE_CLOUD_PROJECT.")
    return project


PROJECT_ID = detect_project_id()
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = GCP_LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_GENAI_USE_ENTERPRISE"] = "TRUE"

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", datefmt="%H:%M:%S", force=True
)
for noisy_logger in ("httpx", "google_adk", "google_genai", "google.auth"):
    logging.getLogger(noisy_logger).setLevel(logging.WARNING)
logger = logging.getLogger("answer_workflow")

print(f"Project: {PROJECT_ID}   Model: {MODEL} ({GCP_LOCATION})")

## Deployment settings

`LOCATION` is where the deployed agent runs; `MODEL_LOCATION` is where it calls Gemini (`global`, set in the configuration cell above). The deployed agent receives `MODEL_LOCATION` through `env_vars`.

In [ ]:
import vertexai
from google.cloud import storage

LOCATION = "us-central1"              # region the Agent Engine runs in
MODEL_LOCATION = GCP_LOCATION         # region the agent calls Gemini in ("global")
STAGING_BUCKET = f"gs://{PROJECT_ID}-agent-staging"

bucket_name = STAGING_BUCKET.removeprefix("gs://")
storage_client = storage.Client(project=PROJECT_ID)
if storage_client.lookup_bucket(bucket_name) is None:
    storage_client.create_bucket(bucket_name, location=LOCATION)
    print(f"Created staging bucket {STAGING_BUCKET}")
else:
    print(f"Using staging bucket {STAGING_BUCKET}")

vertexai.init(project=PROJECT_ID, location=LOCATION, staging_bucket=STAGING_BUCKET)

## 1. Define the agent (from Challenge 4)

In [ ]:
from typing import Literal

from google.adk import Agent
from google.adk.tools import google_search
from google.genai import types
from pydantic import BaseModel, Field


class Intake(BaseModel):
    """The greeter's structured verdict on the user's message."""

    intent: Literal["greeting", "question"] = Field(
        description="'greeting' for hellos, thanks or 'what can you do'; 'question' for anything to answer."
    )
    reply: str = Field(description="A short friendly reply when intent is 'greeting'; otherwise empty.")


greeter_agent = Agent(
    name="greeter_agent",
    model=MODEL,
    description="Welcomes the user and decides whether the message needs the answer team.",
    instruction=(
        "You are the friendly front desk of a question-answering service. Classify the user's "
        "message. If it is only a greeting, thanks, or a question about what you can do, set "
        "intent='greeting' and write a one- or two-sentence welcome that invites a question "
        "(the service researches answers, then fact-checks and polishes them). Otherwise set "
        "intent='question' and leave reply empty."
    ),
    output_schema=Intake,
    output_key="intake",
    generate_content_config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(thinking_budget=0)
    ),
)

search_agent = Agent(
    name="search_agent",
    model=MODEL,
    description="Researches the question with Google Search and writes a first draft.",
    instruction=(
        "Research the user's question with Google Search, then write a complete, practical "
        "first-draft answer.\n\nQuestion: {question}\n\n"
        "Cover the main steps or facts, include specifics (quantities, order of steps, "
        "tools), and list the sources you used at the end."
    ),
    tools=[google_search],
    output_key="draft_answer",
)

critique_agent = Agent(
    name="critique_agent",
    model=MODEL,
    description="Reviews the draft answer and lists concrete improvements.",
    instruction=(
        "You are a demanding editor and fact-checker. Review the draft answer to the question.\n\n"
        "Question: {question}\n\nDraft answer:\n{draft_answer}\n\n"
        "Check accuracy, completeness (missing steps or considerations), safety, organization, "
        "and clarity for a non-expert. Output a numbered list of specific, actionable "
        "improvements, most important first. Do not rewrite the answer yourself. If the draft "
        "is already excellent, say so and suggest only minor polish."
    ),
    output_key="critique",
)

refiner_agent = Agent(
    name="refiner_agent",
    model=MODEL,
    description="Rewrites the draft applying the critique; produces the final answer.",
    instruction=(
        "Rewrite the draft answer so that it applies every improvement in the critique.\n\n"
        "Question: {question}\n\nDraft answer:\n{draft_answer}\n\nCritique:\n{critique}\n\n"
        "Output only the final answer for the user: well organized (short headings or "
        "numbered steps), accurate, and easy to follow. Keep the sources list."
    ),
    output_key="final_answer",
)

## Workflow graph

Function nodes are plain Python. Returning an `Event` lets a node pass `output` to the next node, write `state`, choose a `route`, or send a `message` to the user.

In [ ]:
from google.adk import Context, Event, Workflow


def capture_question(node_input) -> Event:
    """START node: store the user's message (a types.Content) in state for the agents' instructions.

    node_input is deliberately not type-annotated: ADK builds a validator from the
    annotation, and the one for types.Content can't be pickled, which breaks
    deployment to Agent Platform (Challenge 5).
    """
    question = "".join(part.text or "" for part in node_input.parts or []).strip()
    return Event(output=question, state={"question": question})


def route_intent(ctx: Context, node_input: dict) -> Event:
    """Route on the greeter's verdict; questions go on to the answer team."""
    intent = node_input.get("intent", "question")
    logger.info("route_intent -> %s", intent)
    if intent == "greeting":
        return Event(route="greeting", output=node_input.get("reply") or "Hello! Ask me a question.")
    return Event(route="question", output=ctx.state.get("question", ""))


def send_greeting(node_input: str) -> Event:
    """Reply to greetings without running the answer team."""
    return Event(message=node_input)


answer_workflow = Workflow(
    name="AnswerWorkflow",
    description="Greets, then answers questions with a Search -> Critique -> Refine team.",
    edges=[
        ("START", capture_question, greeter_agent, route_intent),
        (route_intent, {"greeting": send_greeting, "question": search_agent}),
        (search_agent, critique_agent),
        (critique_agent, refiner_agent),
    ],
)

# For ADK compatibility (adk web / adk run), the root must be named root_agent.
root_agent = answer_workflow

## Test locally with `AdkApp`

`AdkApp` is the wrapper Agent Platform runs. A graph `Workflow` isn't an agent class, so it goes in through an ADK `App`. Testing this exact object locally catches problems before a several-minute deployment.

In [ ]:
from google.adk.apps import App
from vertexai.agent_engines import AdkApp

app = AdkApp(app=App(name="answer_workflow", root_agent=root_agent))


async def stream(agent_app, question: str, user_id: str = "student") -> None:
    """Sends one question to a local or deployed agent and prints each node's text output."""
    print(f"YOU: {question}\n")
    async for event in agent_app.async_stream_query(user_id=user_id, message=question):
        parts = (event.get("content") or {}).get("parts") or []
        text = "".join(part.get("text") or "" for part in parts).strip()
        if text:
            print(f"===== {event.get('author')} =====\n{text}\n")


await stream(app, "How do I clean out a closet?")

## Deploy to Agent Platform

The requirements pin the same library versions as this notebook. The platform rebuilds the agent from a pickled copy of `app`, and matching versions keep that copy loadable.

In [ ]:
import importlib.metadata

from vertexai import types as vertex_types  # aliased: `types` is google.genai.types in the agent code

REQUIREMENTS = [
    f"google-cloud-aiplatform[agent_engines,adk]=={importlib.metadata.version('google-cloud-aiplatform')}",
    f"google-adk=={importlib.metadata.version('google-adk')}",
    f"cloudpickle=={importlib.metadata.version('cloudpickle')}",
    f"pydantic=={importlib.metadata.version('pydantic')}",
]
print("Requirements:", *REQUIREMENTS, sep="\n  ")

client = vertexai.Client(
    project=PROJECT_ID,
    location=LOCATION,
)

remote_agent = client.agent_engines.create(
    agent=app,
    config={
        "requirements": REQUIREMENTS,
        "staging_bucket": STAGING_BUCKET,
        "env_vars": {"GOOGLE_CLOUD_LOCATION": MODEL_LOCATION},
        "identity_type": vertex_types.IdentityType.AGENT_IDENTITY,
        "display_name": "Answer Workflow Agent",
        "description": "Greets users, then researches, critiques and refines answers to their questions.",
    },
)
print("Deployed:", remote_agent.api_resource.name)

## Test the deployed agent

Same questions, now answered by the copy running on Agent Platform. Each node's output shows the workflow running remotely.

In [ ]:
await stream(remote_agent, "Hi there! What can you do?")

In [ ]:
await stream(remote_agent, "How do I clean out a closet?")

## Clean up (optional)

Deployed agents keep running, and billing, until deleted. Set `DELETE_AGENT = True` to remove it after grading.

In [ ]:
DELETE_AGENT = False

if DELETE_AGENT:
    client.agent_engines.delete(name=remote_agent.api_resource.name, force=True)
    print("Deleted", remote_agent.api_resource.name)
else:
    print("Kept", remote_agent.api_resource.name)